# Task A — Feature ablation multi-seed

**Restart & Run All.**

Pobiera runy z W&B (`TaskA_FEATURE_ABLATION_SEEDS` albo diagnostyczną `TaskA_FEATURE_ABLATION`) i liczy parowane różnice przy tym samym seedzie:

$$\Delta_{emp-top}^{(s)} = AUPRC_{emp}^{(s)} - AUPRC_{top}^{(s)}$$
$$\Delta_{emp-shuf}^{(s)} = AUPRC_{emp}^{(s)} - AUPRC_{shuf}^{(s)}$$

Raport: $\bar{\Delta} \pm SD(\Delta)$.

Wniosek metodyczny: **nie wdrażamy HCR na ślepo** — najpierw potwierdzamy na wielu seedach, że cechy marginalne są słabe / jak silnie zależą od architektury.

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import wandb
from IPython.display import display

warnings.filterwarnings("ignore", category=Warning)

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / "TaskA_FEATURE_ABLATION_comparison.ipynb").exists() else NOTEBOOK_DIR
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "configs").is_dir() and (parent / "src").is_dir():
        REPO_ROOT = parent
        break

OUT_DIR = REPO_ROOT / "outputs" / "taskA_FEATURE_ABLATION_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ENTITY = "politechnika-gnn-thesis"
PROJECT = "politechnika-gnn-thesis"
# Prefer multi-seed group; fall back to diagnostic 6-run group.
GROUPS = ["TaskA_FEATURE_ABLATION_SEEDS", "TaskA_FEATURE_ABLATION"]

api = wandb.Api()
runs = []
used_group = None
for group in GROUPS:
    batch = list(api.runs(f"{ENTITY}/{PROJECT}", filters={"group": group}))
    finished = [r for r in batch if r.state == "finished"]
    print(f"{group}: {len(finished)} finished / {len(batch)} total")
    if finished:
        runs = finished
        used_group = group
        if group == "TaskA_FEATURE_ABLATION_SEEDS" and len(finished) >= 6:
            break

print("Using group:", used_group)

In [ ]:
def cfg_get(cfg, *keys, default=None):
    cur = cfg
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


rows = []
for run in runs:
    cfg = dict(run.config or {})
    summary = dict(run.summary or {})
    rows.append(
        {
            "run_id": run.id,
            "run_name": run.name,
            "model": cfg_get(cfg, "model", "conv_type", default=cfg_get(cfg, "model", "name")),
            "num_layers": cfg_get(cfg, "model", "num_layers"),
            "feat": cfg_get(cfg, "data", "feature_ablation_profile",
                            default=summary.get("feature_ablation_profile")),
            "training_seed": cfg_get(cfg, "training", "seed"),
            "valid_auprc": summary.get("valid_auprc", summary.get("best_valid_AUPRC")),
            "test_auprc": summary.get("test_auprc"),
            "valid_auc": summary.get("valid_auc"),
            "test_auc": summary.get("test_auc"),
            "valid_brier": summary.get("valid_brier"),
            "candidate_fingerprint": summary.get("candidate_fingerprint"),
            "feature_fingerprint": summary.get("feature_fingerprint"),
        }
    )

df = pd.DataFrame(rows)
df["valid_auprc"] = pd.to_numeric(df["valid_auprc"], errors="coerce")
df["test_auprc"] = pd.to_numeric(df["test_auprc"], errors="coerce")
df["training_seed"] = pd.to_numeric(df["training_seed"], errors="coerce").astype("Int64")
display(df.sort_values(["model", "training_seed", "feat"]))

In [ ]:
def paired_deltas(frame: pd.DataFrame, metric: str = "valid_auprc") -> pd.DataFrame:
    rows = []
    for (model, seed), g in frame.groupby(["model", "training_seed"]):
        by_feat = {str(r.feat): float(r[metric]) for _, r in g.iterrows() if pd.notna(r[metric])}
        if "empirical" not in by_feat:
            continue
        emp = by_feat["empirical"]
        row = {
            "model": model,
            "training_seed": int(seed) if pd.notna(seed) else None,
            "auprc_empirical": emp,
            "auprc_topology_only": by_feat.get("topology_only"),
            "auprc_empirical_shuffled": by_feat.get("empirical_shuffled"),
        }
        if "topology_only" in by_feat:
            row["delta_emp_top"] = emp - by_feat["topology_only"]
        if "empirical_shuffled" in by_feat:
            row["delta_emp_shuf"] = emp - by_feat["empirical_shuffled"]
        rows.append(row)
    return pd.DataFrame(rows)


deltas_valid = paired_deltas(df, "valid_auprc")
deltas_test = paired_deltas(df, "test_auprc")
print("Paired deltas on VALID")
display(deltas_valid)
print("Paired deltas on TEST (report only)")
display(deltas_test)

In [ ]:
def summarize_deltas(deltas: pd.DataFrame, split_name: str) -> pd.DataFrame:
    rows = []
    for model, g in deltas.groupby("model"):
        for col, label in [
            ("delta_emp_top", "emp - topology_only"),
            ("delta_emp_shuf", "emp - shuffled"),
        ]:
            if col not in g.columns:
                continue
            vals = pd.to_numeric(g[col], errors="coerce").dropna()
            rows.append(
                {
                    "split": split_name,
                    "model": model,
                    "contrast": label,
                    "n_seeds": int(len(vals)),
                    "mean_delta": float(vals.mean()) if len(vals) else np.nan,
                    "sd_delta": float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
                    "mean_pm_sd": (
                        f"{vals.mean():+.4f} ± {vals.std(ddof=1):.4f}"
                        if len(vals) > 1
                        else (f"{vals.mean():+.4f}" if len(vals) else "n/a")
                    ),
                }
            )
    return pd.DataFrame(rows)


summary = pd.concat(
    [summarize_deltas(deltas_valid, "valid"), summarize_deltas(deltas_test, "test")],
    ignore_index=True,
)
display(summary)

print(
    """
Interpretacja (valid):
  mean Δ(emp-top) >> 0  → cechy wnoszą informację ponad topologię
  mean Δ(emp-top) ≈ 0   → model głównie strukturalny
  mean Δ(emp-shuf) ≈ 0  → konkretne przypisanie cech→węzeł słabo wykorzystywane
  mean Δ(emp-shuf) >> 0 → model używa node-level assignment
"""
)

In [ ]:
df.to_csv(OUT_DIR / "feature_ablation_runs.csv", index=False)
deltas_valid.to_csv(OUT_DIR / "feature_ablation_deltas_valid.csv", index=False)
deltas_test.to_csv(OUT_DIR / "feature_ablation_deltas_test.csv", index=False)
summary.to_csv(OUT_DIR / "feature_ablation_delta_summary.csv", index=False)
print("Wrote CSVs to", OUT_DIR.resolve())